# T04: Evaluation

A RAG system that *seems* to work on a few hand-picked queries can still fail often. Evaluation turns "seems better" into a number you can compare across changes:

```
Change a parameter  ──►  Measure  ──►  Decide based on data
```

## Why Evaluate Systematically?

In T01 the baseline sometimes described the non-existent Lara pallet, cited the non-verified 2021 GWP figure instead of the EPD, or reported tesa's 68% CO₂ reduction without flagging it. Did a new prompt or chunk size fix that, and did it break something else? Only a fixed test set and metrics tell you.

Each pipeline stage fails differently, so metrics target specific stages:

| Stage | If it fails |
|---|---|
| Chunking | Facts split across chunks, tables broken |
| Embedding and retrieval | Relevant chunks not returned or ranked low |
| Generation | Hallucination, ignored context, incomplete answer |

## RAGAS

[RAGAS](https://docs.ragas.io) evaluates RAG pipelines with an LLM as judge. Instead of asking for a 0 to 5 rating, it splits the answer into atomic claims and checks each against the retrieved context:

```
Answer: "The Logypal 1 EPD is registered as S-P-09135 and verified by Bureau Veritas."

  Claim 1: "registered as S-P-09135"   -> supported by context?  ✓
  Claim 2: "verified by Bureau Veritas" -> supported by context?  ✓

  Faithfulness = 2 / 2 = 1.0
```

This catches partial hallucinations, such as a correct figure with an invented verifier.

| Metric | Needs ground truth? | Catches |
|---|---|---|
| `Faithfulness` | No | Claims not supported by the retrieved context |
| `AnswerRelevancy` | No | Off-topic or evasive answers |
| `ContextPrecision` | Yes | Irrelevant chunks ranked above relevant ones |
| `ContextRecall` | Yes | Facts missing from the retrieved context |

The ground truth is the hand-written test set `EVALUATION_QUERIES` in `ki_kompass_rag/evaluation.py`. The judge makes several LLM calls per sample and metric, and has its own biases: the scores are proxies, not absolute truth.

## Setup

Requires `OPENAI_API_KEY` (locally: in `.env.local` at the repo root). Run T01 first: it builds the vector store the later notebooks reuse. RAGAS uses `gpt-4o-mini` as judge.

In [ ]:
import math
import sys

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings as LangChainOpenAIEmbeddings
from loguru import logger
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import AnswerRelevancy, ContextPrecision, ContextRecall, Faithfulness

from conversational_toolkit.evaluation import Evaluator
from conversational_toolkit.evaluation.adapters import evaluate_with_ragas
from ki_kompass_rag.baseline_rag import (
    RETRIEVER_TOP_K,
    SYSTEM_PROMPT,
    build_agent,
    build_llm,
    load_or_build_vector_store,
)
from ki_kompass_rag.evaluation import EVALUATION_QUERIES

logger.remove()
logger.add(sys.stderr, level="WARNING")

vector_store, embedding_model = await load_or_build_vector_store()
agent = build_agent(
    vector_store,
    embedding_model,
    build_llm("openai"),
    top_k=RETRIEVER_TOP_K,
    system_prompt=SYSTEM_PROMPT,
)

# RAGAS's default limit of 3072 output tokens cuts off the judge's JSON on long answers.
ragas_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4o-mini", max_completion_tokens=8192)
)
ragas_embeddings = LangChainOpenAIEmbeddings(model="text-embedding-3-small")


def print_scores(report, queries: list[str]) -> None:
    for name, score in report.summary().items():
        print(f"{name:<20}{score:.3f}")
    print("\n" + "".join(f"{r.metric_name:>20}" for r in report.results) + "  query")
    for i, query in enumerate(queries):
        scores = [r.per_sample_scores[i] for r in report.results]
        cells = "".join(
            f"{'n/a' if math.isnan(s) else format(s, '.2f'):>20}" for s in scores
        )
        print(f"{cells}  {query[:60]}")

## Part 1: Generation Quality (No Ground Truth)

`Evaluator.build_samples_from_agent()` runs the agent on every test query and stores the answer, the retrieved chunks and the reference answer. Faithfulness and AnswerRelevancy only use the query, answer and chunks. This takes a few minutes.

In [ ]:
queries = [q["query"] for q in EVALUATION_QUERIES]
samples = await Evaluator.build_samples_from_agent(
    agent=agent,
    queries=queries,
    ground_truth_answers=[q["ground_truth_answer"] for q in EVALUATION_QUERIES],
)

report_generation = evaluate_with_ragas(
    samples=samples,
    metrics=[Faithfulness(), AnswerRelevancy(strictness=1)],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)
print_scores(report_generation, queries)

Low faithfulness on a query means the answer contains claims the retrieved chunks do not support. Read that answer: `samples[i].answer`.

## Part 2: Retrieval Quality (Ground Truth Required)

Generation metrics say nothing about whether the retriever found the *right* chunks. ContextPrecision and ContextRecall compare the retrieved chunks with the reference answers stored in `samples`.

| Metric | 1.0 means | Low means |
|---|---|---|
| ContextPrecision | All retrieved chunks are relevant, best ones first | Noisy or badly ranked chunks |
| ContextRecall | Every fact of the reference answer is in the context | Retriever misses key chunks |

In [ ]:
report_retrieval = evaluate_with_ragas(
    samples=samples,
    metrics=[ContextPrecision(), ContextRecall()],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)
print_scores(report_retrieval, queries)

## Summary

| Metric | Ground truth? | Tells you |
|---|---|---|
| `Faithfulness` | No | Are answers grounded in the retrieved context? |
| `AnswerRelevancy` | No | Do answers address the question? |
| `ContextPrecision` | Yes | Are retrieved chunks relevant and well ranked? |
| `ContextRecall` | Yes | Does the context contain all needed facts? |

Low generation scores: look at the prompt or the LLM. Low retrieval scores: look at chunking, embeddings or `top_k` (T02, T07). T05 builds a larger test set.